# Aula 02 — Componentes do LangGraph: StateGraph, Estado e Ferramentas

Este notebook apresenta os **blocos fundamentais** para construir um agente ReAct com LangGraph e Google Gemini:

- **`AgentState`:** schema de estado compartilhado entre todos os nós do grafo, usando `Annotated[..., operator.add]` como reducer para acumulação imutável do histórico de mensagens.
- **`StateGraph`:** grafo dirigido e tipado que orquestra o fluxo de execução entre o LLM e as ferramentas (padrão ReAct).
- **`bind_tools`:** mecanismo que expõe os schemas das ferramentas ao modelo, habilitando geração de `tool_calls` estruturadas.
- **`TavilySearch`:** ferramenta de busca na web integrada ao agente via LangChain.
- **`graph.stream()` vs `graph.invoke()`:** duas formas de executar o grafo — streaming em tempo real ou resultado completo ao final.
- **System prompt com data dinâmica:** técnica para garantir que o agente use a data atual em queries de busca sobre "hoje".

## 1. Esquemas de Estado — Exploração Inicial

Antes de qualquer import, o notebook explora dois formatos alternativos para o schema do estado do grafo. Isso é didático: mostra que o `AgentState` pode ter diferentes formas dependendo da arquitetura desejada.

### O que é um schema de estado?

No LangGraph, todo nó do grafo lê e escreve em um dicionário tipado chamado **estado**. O schema define os campos disponíveis e como cada campo é atualizado quando um nó retorna valores.

### O reducer `operator.add`

```python
messages: Annotated[list, operator.add]
```

Sem o reducer, cada nó que retornasse `{'messages': [...]}` **substituiria** todo o histórico. Com `operator.add`, os valores são **concatenados** — preservando o histórico completo de mensagens ao longo de toda a execução do grafo.

In [ ]:
# Esquema mínimo — versão exploratória (NÃO é usada no grafo final)
# Nota: 'BaseMassage' é um typo intencional do notebook original (deveria ser BaseMessage).
# Esta célula serve para introduzir o conceito de reducer antes de qualquer import.
# operator.add como reducer: em vez de substituir 'messages', cada atualização
# ADICIONA novas mensagens à lista existente, preservando o histórico completo.
class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMassage], operator.add]  # typo original: BaseMassage

In [ ]:
# Schema mais rico — versão exploratória baseada no LangChain Agent clássico (NÃO usada no grafo final)
# Mostra uma alternativa ao formato de 'apenas mensagens': separar entrada, histórico,
# resultado do agente e passos intermediários em campos distintos.
# 'agent_outcome: Union[AgentAction, AgentFinish, None]' = pode estar em execução, finalizado ou aguardando.
# 'intermediate_steps' usa operator.add para acumular pares (ação, observação) ao longo do loop.
class AgentState(TypedDict):
  input: str                                                           # Pergunta do usuário (imutável)
  chat_history: list[BaseMessage]                                      # Histórico anterior (não acumulado)
  agent_outcome: Union[AgentAction, AgentFinish, None]                 # Estado atual do agente
  intermediate_steps: Annotated[list[tuple[AgentAction, str]], operator.add]  # Acumula (ação, resultado)

## 2. Instalação das Dependências

Instalamos os pacotes necessários para o notebook. Note que alguns pacotes são instalados mais de uma vez com `-U` (upgrade) — isso é comum em notebooks exploratórios onde a versão exata ainda está sendo determinada.

| Pacote | Finalidade |
|--------|-----------|
| `google-generativeai` | SDK oficial da Google para o Gemini |
| `langchain-google-genai` | Wrapper LangChain para o Gemini (`ChatGoogleGenerativeAI`) |
| `langchain-community` | Ferramentas da comunidade LangChain (inclui `TavilySearchResults`) |
| `langgraph` | Framework de grafos de agentes |
| `python-dotenv` | Carrega variáveis de ambiente do arquivo `.env` |

In [ ]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph
%pip install -U langgraph langchain-community
%pip install python-dotenv

## 3. Imports

Importamos os blocos principais do LangGraph e LangChain necessários para construir o agente:

| Import | Pacote | Papel |
|--------|--------|-------|
| `StateGraph, END` | `langgraph` | Núcleo do grafo: estrutura dirigida com estado tipado e nó terminal |
| `TypedDict, Annotated, List` | `typing` | Tipagem estática do estado e anotações de reducer |
| `operator` | stdlib | Fornece `operator.add` como reducer do estado |
| `google.generativeai` | `google-generativeai` | SDK Google (necessário para configuração da API key) |
| `ChatGoogleGenerativeAI` | `langchain-google-genai` | Wrapper LangChain para o Gemini |
| `AnyMessage, SystemMessage, HumanMessage, ToolMessage` | `langchain-core` | Tipos de mensagem do protocolo LangChain |
| `TavilySearchResults` | `langchain-community` | Ferramenta de busca web (versão community, substituída na refatoração) |

In [ ]:
from langgraph.graph import StateGraph, END  # StateGraph: grafo tipado; END: nó terminal especial
from typing import TypedDict, Annotated, List
import operator  # operator.add será usado como reducer do estado

import google.generativeai as genai  # SDK Google — necessário para inicializar as credenciais
from langchain_google_genai import ChatGoogleGenerativeAI  # Wrapper LangChain para o Gemini

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
# AnyMessage: tipo union para qualquer mensagem LangChain (aceita HumanMessage, AIMessage, etc.)
# SystemMessage: instrução inicial de comportamento do agente (não persistida no estado)
# HumanMessage: entrada do usuário
# ToolMessage: resultado da execução de uma ferramenta, vinculado por tool_call_id

from langchain_community.tools.tavily_search import TavilySearchResults
# TavilySearchResults: versão community da ferramenta Tavily — será substituída por
# langchain_tavily.TavilySearch na refatoração posterior (célula 12)

In [ ]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env para os.environ
load_dotenv()

# langchain-google-genai espera a chave em GOOGLE_API_KEY;
# fazemos o mapeamento aqui para manter o nome GEMINI_API_KEY no .env
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## 5. Ferramenta de Busca — TavilySearchResults

Instanciamos a ferramenta de busca que será fornecida ao agente. O objeto retornado é um `BaseTool` LangChain — qualquer ferramenta compatível com essa interface pode ser usada no lugar de Tavily (DuckDuckGo, Brave, etc.).

O campo `.name` é crítico: é o identificador que o LLM usa ao gerar `tool_calls`. Quando o Gemini decide fazer uma busca, ele emite `{"name": "tavily_search_results_json", "args": {...}}` — e a classe `Agent` usa exatamente esse nome para fazer o lookup no dicionário `self.tools`.

In [ ]:
# max_results=4: limita a 4 resultados por busca para controlar custo de tokens.
# O atributo .name é o identificador que o LLM usa nas tool_calls —
# deve coincidir com a chave usada no dicionário self.tools dentro da classe Agent.
tool = TavilySearchResults(max_results=4)
print(type(tool))   # Confirma que é um BaseTool LangChain
print(tool.name)    # Imprime 'tavily_search_results_json'

## 6. AgentState — Versão Final

Após explorar as versões alternativas (células 1 e 2), definimos o schema definitivo do estado. A simplificação é intencional: manter apenas `messages` com o reducer `operator.add` é suficiente para o padrão ReAct — todas as interações (perguntas, respostas, resultados de ferramentas) são representadas como mensagens.

```python
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
```

**Por que `AnyMessage` e não `BaseMessage`?** `AnyMessage` é um alias de tipo que abrange todos os subtipos (`HumanMessage`, `AIMessage`, `ToolMessage`, etc.) sem precisar de importações adicionais. Na refatoração posterior, a versão usa `List[BaseMessage]` — ambas são equivalentes em funcionalidade.

In [ ]:
# Versão final do AgentState — schema simples com um único campo acumulativo.
# list[AnyMessage] aceita qualquer subtipo de mensagem LangChain.
# operator.add garante que cada nó ADICIONA mensagens ao histórico em vez de sobrescrever.
class AgentState(TypedDict): 
    messages: Annotated[list[AnyMessage], operator.add]

## 7. Classe Agent — Construção do Grafo ReAct

A classe `Agent` encapsula toda a lógica de construção e execução do grafo. O padrão implementado é o **ReAct** (Reasoning + Acting): o LLM raciocina e decide se precisa agir (chamar ferramentas), e o ciclo se repete até a resposta final.

```
  [entrada] ──► [nó: llm] ──► existe tool_call? ──► SIM ──► [nó: action] ──┐
                    ▲                                                         │
                    └─────────────────────────────────────────────────────────┘
                                    │ NÃO
                                    ▼
                                   END
```

### Decisões de design

| Componente | Decisão | Motivo |
|------------|---------|--------|
| `bind_tools(tools)` | Vincula tools ao modelo | Gera schemas JSON das ferramentas para o LLM produzir `tool_calls` estruturadas |
| `{t.name: t for t in tools}` | Dict nome→ferramenta | Lookup O(1) por nome ao executar `tool_calls` |
| `SystemMessage` injetado dinamicamente | Não persiste no estado | Evita duplicar o prompt em cada checkpoint do histórico |
| `"bad tool name, retry"` | Fallback de erro | Permite que o LLM corrija o nome da ferramenta na próxima iteração |
| `graph.compile()` sem checkpointer | Sem persistência | Nesta versão inicial, cada execução começa com estado zerado (sem memória entre chamadas) |

In [ ]:
class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system  # Prompt de sistema armazenado — injetado dinamicamente em cada chamada
        
        # Inicializa o grafo com o schema de estado AgentState
        graph = StateGraph(AgentState)
        
        # Adiciona os dois nós do grafo: LLM e executor de ações
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        
        # Aresta condicional: se o LLM gerou tool_calls → vai para 'action';
        # caso contrário → termina em END (resposta final para o usuário)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        
        # Após executar as ferramentas, sempre retorna ao LLM
        # (o LLM processa os resultados e decide se precisa de mais ações — loop ReAct)
        graph.add_edge("action", "llm")
        
        # Ponto de entrada: toda execução começa pelo nó 'llm'
        graph.set_entry_point("llm")
        
        # Compila sem checkpointer: estado não persiste entre chamadas separadas
        self.graph = graph.compile()
        
        # Dict nome→ferramenta para lookup eficiente em take_action
        self.tools = {t.name: t for t in tools}
        
        # bind_tools: registra os schemas JSON das ferramentas no modelo.
        # Sem isso, o Gemini não sabe que pode gerar tool_calls para essas ferramentas.
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        """Função de roteamento: verifica se a última mensagem do LLM contém tool_calls."""
        result = state['messages'][-1]
        return len(result.tool_calls) > 0  # True → executa ferramentas; False → encerra

    def call_gemini(self, state: AgentState):
        """Nó 'llm': invoca o Gemini com o histórico completo de mensagens."""
        messages = state['messages']
        
        # Injeta o system prompt no início do histórico a cada chamada.
        # É feito dinamicamente (não armazenado no estado) para evitar duplicação no histórico.
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        
        # Retorna a nova mensagem para ser ADICIONADA ao estado via operator.add
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        """Nó 'action': executa todas as tool_calls da última mensagem do LLM."""
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:
                # Fallback para nomes de ferramenta inválidos — retorna mensagem de erro
                # para que o LLM tente novamente com o nome correto
                print("\n ....bad tool name....")
                result = "bad tool name, retry"
            else:
                result = self.tools[t['name']].invoke(t['args'])
            # ToolMessage vincula o resultado à tool_call pelo 'id' —
            # o LLM usa essa vinculação para rastrear qual resultado pertence a qual chamada
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

## 8. Instanciação do Agente

Instanciamos o agente com o modelo Gemini, a ferramenta Tavily e o system prompt. O `temperature=0` é uma decisão deliberada: em agentes que precisam tomar decisões determinísticas sobre quando e como usar ferramentas, a aleatoriedade introduzida por temperaturas maiores aumenta a variância do comportamento sem benefício claro.

In [ ]:
# System prompt: define o comportamento do agente e as regras para uso das ferramentas.
# A instrução de múltiplas chamadas é importante para permitir que o agente pesquise
# em etapas (ex: buscar primeiro o país, depois o PIB).
prompt = """Você é um assistente de pesquisa inteligente. Use o mecanismo de busca para procurar informações. \
Você tem permissão para fazer múltiplas chamadas (seja em conjunto ou em sequência). \
Procure informações apenas quando tiver certeza do que você quer. \
Se precisar pesquisar alguma informação antes de fazer uma pergunta de acompanhamento, você tem permissão para fazer isso!
"""

# temperature=0: maximiza consistência — o modelo escolherá sempre a ação mais provável,
# reduzindo variações aleatórias na seleção de ferramentas e na síntese de respostas
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Instancia o agente com o modelo, a ferramenta Tavily e o system prompt
abot = Agent(model, [tool], system=prompt)

## 9. Visualizando o Grafo

O LangGraph oferece dois métodos para visualizar a estrutura do grafo compilado:

- **`draw_mermaid()`:** retorna uma string no formato [Mermaid](https://mermaid.js.org/), que pode ser visualizada em editores como mermaid.live ou diretamente no GitHub/VSCode.
- **`draw_mermaid_png()`:** gera um PNG diretamente (requer dependências extras). Usamos try/except para degradar graciosamente se as dependências não estiverem disponíveis.

In [ ]:
mermaid_code = abot.graph.get_graph().draw_mermaid()
print(mermaid_code)

In [ ]:
from IPython.display import Image, display

try:
    
    image_data = abot.graph.get_graph().draw_mermaid_png()
    display(Image(data=image_data))

except Exception as e:
    print(f"Erro ao tentar gerar PNG do Mermaid: {e}")
    print("\nCertifique-se de que a sua versão do LangGraph possui o método `.draw_mermaid_png()`.")
    print("Como alternativa, use `.draw_mermaid()` para obter a string e visualizar externamente.")

## 10. Instalação langchain-tavily

O pacote `langchain-tavily` é o pacote **dedicado e mantido pela Tavily** para integração com LangChain. É preferível ao `langchain-community` porque recebe atualizações diretamente da Tavily e expõe a classe `TavilySearch` (mais simples que `TavilySearchResults`).

A célula seguinte refatora o agente usando este pacote.

In [ ]:
%pip install -U langchain-tavily

## 11. Agent Refatorado com TavilySearch

Refatoração do agente usando o pacote `langchain-tavily` dedicado. As mudanças principais em relação à versão anterior:

| Aspecto | Versão anterior | Versão refatorada |
|---------|----------------|-------------------|
| Pacote da ferramenta | `langchain-community` | `langchain-tavily` |
| Classe da ferramenta | `TavilySearchResults` | `TavilySearch` |
| Tipo do estado | `AnyMessage` | `List[BaseMessage]` |
| Nome da var. de loop | `t` | `t_call` (mais descritivo) |

A lógica do grafo ReAct permanece idêntica — a refatoração é apenas de dependências e clareza de código.

In [ ]:
# Refatoração: usa BaseMessage (classe base) em vez de AnyMessage (tipo union)
# ambos funcionam com operator.add, mas BaseMessage é mais adequado para tipagem genérica
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t_call in tool_calls:
            # t_call (vs t da versão anterior): nome mais descritivo, deixa claro que é uma tool_call
            print(f"Calling tool: {t_call['name']} with args: {t_call['args']}")
            if t_call['name'] not in self.tools:
                print("\n ....bad tool name....")
                result = "bad tool name, retry"
            else:
                result = self.tools[t_call['name']].invoke(t_call['args'])
            results.append(ToolMessage(tool_call_id=t_call['id'], name=t_call['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

## 12. Primeira Execução — Prompt sem Data

Primeira execução real do agente com `graph.stream()`. O prompt **não inclui a data atual**, o que pode fazer o agente buscar por "hoje" sem contexto temporal preciso — potencialmente recuperando resultados desatualizados.

### Por que `graph.stream()` e não `graph.invoke()`?

| Método | Comportamento | Quando usar |
|--------|--------------|-------------|
| `graph.stream()` | Emite um evento a cada transição de nó (gerador) | Depuração, visualização em tempo real do raciocínio |
| `graph.invoke()` | Bloqueia até o fim, retorna estado final completo | Quando só o resultado final importa |

O `stream()` é ideal aqui porque permite ver o raciocínio do agente nó a nó (qual ferramenta foi chamada, com quais argumentos, o que foi retornado).

In [ ]:
prompt = """Você é um assistente de pesquisa inteligente. Use o mecanismo de busca para procurar informações. \
Você tem permissão para fazer múltiplas chamadas (seja em conjunto ou em sequência). \
Procure informações apenas quando tiver certeza do que você quer. \
Se precisar pesquisar alguma informação antes de fazer uma pergunta de acompanhamento, você tem permissão para fazer isso!
"""
# Instancia modelo e ferramenta separadamente para facilitar substituição em células seguintes
model_instance = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tool_instance = TavilySearch(max_results=4)  # Ferramenta do pacote langchain-tavily

abot = Agent(model_instance, [tool_instance], system=prompt)

messages = [HumanMessage(content="Como está o tempo em São Paulo hoje?")]

print("Iniciando interação do agente:")
final_result_state = None

# graph.stream() retorna um gerador — cada iteração 's' é um evento de nó
# Formato: {'nome_do_no': {'messages': [lista_de_mensagens]}}
for s in abot.graph.stream({"messages": messages}):
    print(s)
    print("---")
    final_result_state = s  # Guarda o último evento para extrair a resposta final

print("\nResultado Final:")
if final_result_state and 'llm' in final_result_state and final_result_state['llm']['messages']:
    print(final_result_state['llm']['messages'][-1].content)
else:
    print("Nenhum resultado final ou resultado inesperado.")

## 13. Prompt com Data Atual — Solução para Consultas "Hoje"

O problema da célula anterior: o agente pesquisa "tempo em São Paulo hoje" sem saber a data atual. O Gemini pode usar dados de cache ou indexação desatualizada.

**Solução:** injetar `date.today()` no system prompt via f-string. Ao incluir a data explicitamente, o agente é instruído a usá-la na query de busca — tornando a consulta à API Tavily específica e determinística.

```python
current_date = date.today().strftime("%d/%m/%Y")
prompt = f"...A data atual é {current_date}. Inclua '{current_date}' na sua consulta..."
```

Essa abordagem é superior a deixar o LLM inferir "hoje" porque: (1) evita que o modelo use a data do seu treinamento, (2) gera queries de busca mais precisas com datas explícitas.

In [ ]:
from datetime import date
current_date = date.today().strftime("%d/%m/%Y")  # Captura a data no momento da execução da célula

# f-string com current_date: o sistema prompt é personalizado com a data atual
# em vez de ser estático. Isso resolve o problema de ambiguidade temporal do "hoje".
prompt = f"""Você é um assistente de pesquisa inteligente e altamente atualizado. \
Sua principal prioridade é encontrar as informações mais RECENTES e em TEMPO REAL sempre que possível. \
A data atual é {current_date}. \
Ao buscar sobre o tempo ou eventos que se referem a "hoje" ou "agora", \
você DEVE **incluir a data atual '{current_date}' na sua consulta para a ferramenta de busca**. \
Por exemplo, se a pergunta é "tempo em cidade x hoje", a consulta para a ferramenta deve ser "tempo em cidade x {current_date}". \
Ignore ou descarte informações que claramente se refiram a datas passadas ou futuras ao responder perguntas sobre "hoje". \
Use o mecanismo de busca para procurar informações, sempre buscando o 'hoje' ou o 'agora' quando o contexto indicar. \
Você tem permissão para fazer múltiplas chamadas (seja em conjunto ou em sequência). \
Procure informações apenas quando tiver certeza do que você quer. \
Se precisar pesquisar alguma informação antes de fazer uma pergunta de acompanhamento, você tem permissão para fazer isso!
""" 

model_instance = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tool_instance = TavilySearch(max_results=4)
# Recria o agente com o novo prompt que inclui a data —
# necessário porque o prompt é passado no __init__ e fica fixo após a criação
abot = Agent(model_instance, [tool_instance], system=prompt)

user_query = "Como está o tempo em São Paulo hoje?"

messages = [HumanMessage(content=user_query)]

print("Iniciando interação do agente:")
final_result_state = None
for s in abot.graph.stream({"messages": messages}):
    print(s)
    print("---")
    final_result_state = s

print("\nResultado Final:")
if final_result_state and 'llm' in final_result_state and final_result_state['llm']['messages']:
    print(final_result_state['llm']['messages'][-1].content)
else:
    print("Nenhum resultado final ou resultado inesperado.")

### Variações Temporais — "Amanhã" e "Ontem"

As duas células seguintes testam se o agente consegue **raciocinar sobre datas relativas** quando o system prompt contém a data atual:

- **"amanhã":** o agente deve calcular `current_date + 1 dia` e incluir isso na query de busca de previsão do tempo.
- **"ontem":** o agente deve calcular `current_date - 1 dia` e buscar dados históricos meteorológicos.

Este é um teste de capacidade de raciocínio temporal do Gemini — o modelo recebe a data atual como texto no prompt e deve fazer aritmética de datas implicitamente ao formular as queries.

In [ ]:
# Reutiliza o mesmo agente (abot) da célula anterior — que já tem a data no prompt.
# Apenas a pergunta do usuário muda. O agente deve inferir "amanhã" = current_date + 1 dia
# e incluir isso na query de busca.
user_query_tomorrow = "Como está o tempo em São Paulo amanhã?"

messages_tomorrow = [HumanMessage(content=user_query_tomorrow)]

print("\n--- Iniciando interação do agente para amanhã ---")
final_result_state_tomorrow = None
for s in abot.graph.stream({"messages": messages_tomorrow}):
    print(s)
    print("---")
    final_result_state_tomorrow = s

print("\n--- Resultado Final para amanhã ---")
if final_result_state_tomorrow and 'llm' in final_result_state_tomorrow and final_result_state_tomorrow['llm']['messages']:
    print(final_result_state_tomorrow['llm']['messages'][-1].content)
else:
    print("Nenhum resultado final ou resultado inesperado para amanhã.")

In [ ]:
from langchain_core.messages import HumanMessage  # Re-import para garantir disponibilidade

# "ontem": testa se o agente com a data atual consegue calcular a data anterior
# e formular uma query de dados históricos de meteorologia
user_query_tomorrow = "Como foi o tempo em São Paulo ontem?" 

messages_tomorrow = [HumanMessage(content=user_query_tomorrow)]

print("\n--- Iniciando interação do agente ---")
final_result_state_tomorrow = None
for s in abot.graph.stream({"messages": messages_tomorrow}):
    print(s)
    print("---")
    final_result_state_tomorrow = s

print("\n--- Resultado Final  ---")
if final_result_state_tomorrow and 'llm' in final_result_state_tomorrow and final_result_state_tomorrow['llm']['messages']:
    print(final_result_state_tomorrow['llm']['messages'][-1].content)
else:
    print("Nenhum resultado final ou resultado inesperado.")

## 14. Usando graph.invoke() — Resultado Completo

Vamos chamar novamente o agente, perguntando: "Qual é o tempo em SP hoje?" Dessa vez, vamos olhar em detalhes o que o agente buscou e como o fez.

Diferença chave em relação ao `stream()`:
- `graph.invoke()` aguarda a execução completa e retorna o **estado final** como um dicionário
- `result['messages']` contém **todas as mensagens** acumuladas via `operator.add`: HumanMessage inicial, AIMessage(s) com tool_calls, ToolMessage(s) com resultados, e AIMessage final com a resposta
- `result['messages'][-1].content` extrai apenas a resposta final do LLM

In [ ]:
messages = [HumanMessage(content="Como está o tempo em São Paulo hoje?")] 
# graph.invoke(): bloqueia até o final e retorna o estado completo.
# Útil quando queremos ver todas as mensagens acumuladas (incluindo tool_calls e resultados).
result = abot.graph.invoke({"messages": messages})

In [ ]:
# result['messages'][-1]: a última mensagem é sempre a resposta final do LLM (AIMessage).
# As mensagens anteriores contêm: HumanMessage, AIMessage com tool_calls, ToolMessages.
# Graças ao operator.add, todas estão preservadas no estado.
result['messages'][-1].content

### Consulta Multi-Cidade

Esta consulta testa a capacidade de **múltiplas tool_calls paralelas**: o agente pode decidir buscar por SP e RJ simultaneamente em uma única iteração, ou fazer duas buscas sequenciais. O LLM com `bind_tools` tem autonomia para escolher a estratégia mais eficiente.

In [ ]:
messages = [HumanMessage(content="Como está o tempo em São Paulo e no Rio de Janeiro hoje?")]
result = abot.graph.invoke({"messages": messages})

In [ ]:
# result['messages'][-1]: extrai a resposta final da consulta SP+RJ.
# Graças ao operator.add, o estado contém todas as mensagens da execução.
result['messages'][-1].content

## 15. Consultas sobre o Passado — Múltiplas Buscas Sequenciais

Esta consulta complexa testa a capacidade de **múltiplas buscas sequenciais** do agente: para responder sobre Copa do Mundo de 1998, PIB do país campeão em 1998 e PIB atual, o agente provavelmente precisará de 3 ou mais chamadas à Tavily.

A diferença em relação às consultas anteriores: aqui usamos `current_state.update(s)` para acumular o estado de todos os nós, em vez de apenas guardar o último evento `final_result_state = s`. Isso é necessário porque queremos o estado final consolidado após todos os nós terminarem, não apenas o último evento emitido.

In [ ]:
query_passado = "Qual país sediou a Copa do Mundo de futebol em 1998? Quem foi o campeão e qual o placar da final? \
Qual era o Produto Interno Bruto (PIB) desse país no ano da Copa e qual é o PIB atual (últimos dados disponíveis, como 2023 ou 2024)? \
Qual a capital desse país e qual sua moeda atual? Responda a cada pergunta separadamente."
messages = [HumanMessage(content=query_passado)]

print("\nIniciando interação do agente para pergunta sobre o passado:")

# current_state.update(s): acumula o estado de todos os nós durante o streaming.
# Necessário porque stream() emite um evento por vez — o estado final só está
# disponível após o loop completar. Diferente de invoke() que retorna tudo de uma vez.
current_state = {}
for s in abot.graph.stream({"messages": messages}):
    current_state.update(s)
    print(s)
    print("---")

print("\n--- Resultado Final para o Passado ---")
if 'llm' in current_state and 'messages' in current_state['llm'] and current_state['llm']['messages']:
    final_message_content = current_state['llm']['messages'][-1].content
    print(final_message_content)
else:
    print("Nenhum resultado final ou resultado inesperado. Verifique os logs acima.")

In [ ]:
print("\n--- Agente de Pesquisa Interativo ---")
print("Digite sua pergunta ou 'sair' para encerrar.")

while True:
    user_input = input("\nVocê: ")  # Bloqueia até o usuário pressionar Enter
    if user_input.lower() == "sair":
        print("Agente: Encerrando a conversa. Até logo!")
        break

    messages = [HumanMessage(content=user_input)]

    print("Agente: Pensando e buscando...")
    try:
        # Cada iteração do loop while cria um novo contexto de execução —
        # sem memória entre perguntas (sem checkpointer = sem estado persistido)
        current_state = {}
        for s in abot.graph.stream({"messages": messages}):
            current_state.update(s)

        print("\nAgente:")

        # Extrai a resposta final do nó 'llm' após o streaming completar
        if 'llm' in current_state and 'messages' in current_state['llm'] and current_state['llm']['messages']:
            final_message = current_state['llm']['messages'][-1]
            if hasattr(final_message, 'content'):
                print(final_message.content)
            else:
                print("Não foi possível extrair o conteúdo da resposta final do LLM.")
        else:
            print("Não foi possível obter uma resposta do agente para esta pergunta.")

    except Exception as e:
        # Captura erros de rede, API ou lógica do grafo sem encerrar o loop
        print(f"Agente: Ocorreu um erro durante a execução: {e}")
        print("Tente novamente ou digite 'sair'.")

print("\n--- Conversa Encerrada ---")

## 16. Interface Interativa — Loop de Conversação

Interface de linha de comando para interação em tempo real com o agente. Cada pergunta é processada de forma independente — **sem memória entre perguntas** (sem checkpointer).

Pontos importantes desta implementação:

| Aspecto | Detalhe |
|---------|---------|
| `while True` com `break` | Loop infinito encerrado por input do usuário — padrão REPL |
| `try/except` global | Captura erros de API ou grafo sem encerrar o loop |
| Sem `thread_id` | Cada pergunta é uma execução isolada — sem histórico compartilhado |
| `current_state.update(s)` | Acumula estado de todos os nós para extrair resposta final do nó `llm` |

Para adicionar memória entre perguntas, seria necessário manter uma lista de mensagens fora do loop e passá-la acumulada a cada `graph.stream()` — ou usar `SqliteSaver` com `thread_id` fixo (ver Aula 04).